# Baseline Model

In [143]:
import sqlite3
import numpy as np

# Our classes are the integers from 10 to 33
board_classes = list(range(10, 34))
board_grades = [
    "4a", "4b", "4c", "5a", "5b", "5c", "6a", "6a+", "6b", "6b+", "6c", "6c+", "7a", "7a+", "7b", "7b+",
    "7c", "7c+", "8a", "8a+", "8b", "8b+", "8c", "8c+"
]

# Convert from difficulty number to board grade
grade_map = dict(zip(board_classes, board_grades))

# Connect to databases
kilter_conn = sqlite3.connect('../data/kilter.sqlite3')
climbs_conn = sqlite3.connect('../data/climbs.sqlite3')
kilter_cursor = kilter_conn.cursor()
climbs_cursor = climbs_conn.cursor()

# Get the list of climbs we are working with (>= 5 ascents, Kilter Board Original (layout 10))
climbs_cursor.execute("""SELECT currentDifficultyId FROM climb_stats
                                  INNER JOIN climbs
                                  ON climb_stats.climbUuid = climbs.climbUuid
                                  WHERE ascentCount >= 5
                                  AND climb_stats.product_layout_uuid = '10'
                                  AND currentDifficultyId >= 10;""")

# Extract grade data
rows = climbs_cursor.fetchall()
climb_grades = np.array(list(map(lambda x: x[0], rows)))
print(f'The first 10 climbs are graded: {climb_grades[:10]} or {list(map(lambda x: grade_map[x], climb_grades))[:10]}.')

The first 10 climbs are graded: [14. 19. 23. 22. 17. 17. 22. 15. 18. 22.] or ['5b', '6b+', '7a+', '7a', '6a+', '6a+', '7a', '5c', '6b', '7a'].


In [144]:
# The baseline model predicts the most common class
climbs_cursor.execute("""SELECT currentDifficultyId, COUNT(*) FROM climb_stats
                                WHERE ascentCount >= 5
                                AND climb_stats.product_layout_uuid = '10'
                                AND currentDifficultyId >= 10
                                GROUP BY currentDifficultyId;
                      """)
grade_counts = np.array(climbs_cursor.fetchall())
print(f'Ex: There are {int(grade_counts[0][1])} climbs graded {grade_map[grade_counts[0][0]]}')

# Find most common class
argmax = np.argmax(grade_counts, axis=0)
most_common = argmax[0]
num_most_common = grade_counts[argmax[1]][1]

print(f'The most common class is {grade_map[most_common]} with {int(num_most_common)} climbs.')

# Get final probability distribution
total_climbs = len(climb_grades)
dist = np.array(list(map(lambda x: x[1] / total_climbs, grade_counts)))

print(f'Probability distribution: {dist}')
print(f'The sum of the {len(dist)} probabilities in the distribution is {dist.sum()}')

Ex: There are 473 climbs graded 4a
The most common class is 7a with 4045 climbs.
Probability distribution: [1.15772469e-02 2.54552575e-02 2.68993538e-02 4.03857451e-02
 4.62110828e-02 5.73722342e-02 8.29009203e-02 6.79704327e-02
 7.82993930e-02 6.38339534e-02 8.04043470e-02 6.19492853e-02
 9.90062659e-02 7.35020560e-02 6.03828079e-02 3.42911690e-02
 3.95290777e-02 2.61161151e-02 1.77697278e-02 6.36381437e-03
 1.19933425e-03 2.44762091e-05 2.44762091e-05]
The sum of the 23 probabilities in the distribution is 1.001468572547484


In [145]:
class BaselineModel:
    def __init__(self):
        self.distribution = dist
    
    def forward(self, X: np.ndarray):
        # Output the probability distribution of grades for each input in the batch
        return np.array([self.distribution] * X.shape[0])

baseline = BaselineModel()

# Use our list of climb grades as input
result = baseline.forward(climb_grades)
acc = 0
for i in range(len(climb_grades)):
    pred = np.argmax(result[i]) + 10 # add 10 to convert from [0, 22] to [10, 32]
    if pred == climb_grades[i]:
        acc += 1
        
print(f"Baseline accuracy: {acc} / {len(climb_grades)} = {acc / len(climb_grades):.3f}.")

# CCE Loss
import torch
from torch.nn import CrossEntropyLoss

# Get one-hot encoding of the climb grades
one_hot = np.array([[0] * len(dist)] * len(climb_grades))
for i in range(len(climb_grades)):
    one_hot[i, int(climb_grades[i]) - 10] = 1

loss = CrossEntropyLoss()
one_hot_tensor = torch.tensor(one_hot, dtype=torch.float32)
result_tensor = torch.tensor(result, dtype=torch.float32)
print(f"CCE loss = {loss(result_tensor, one_hot_tensor)}")

Baseline accuracy: 4043 / 40856 = 0.099.
CCE loss = 3.1163277626037598


In [146]:
# Close databases
kilter_cursor.close()
climbs_cursor.close()